# Notebook 5: Macro Analysis

So far we have shown that the value premium exists, has weakened over time, 
and generates alpha beyond the Fama-French factors for BM.

Now we ask: WHY has value underperformed in recent years?

One popular explanation is interest rates. When interest rates fall, investors 
are willing to pay more for future growth, which benefits growth stocks more 
than value stocks. Growth companies like Amazon and Google derive most of their 
value from cash flows far in the future. When you discount those future cash flows 
at a lower rate, they become worth much more today.

We test this by downloading real interest rate and market volatility data from 
FRED (the Federal Reserve's data repository) and running regressions to see if 
these macro variables explain the value spread.

Variables we will use:
- **GS10:** 10-year Treasury yield — our main interest rate measure
- **VIXCLS:** The VIX — measures market fear/volatility

In [1]:
!pip install fredapi


[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: pip install --upgrade pip


In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import statsmodels.api as sm
from fredapi import Fred

from dotenv import load_dotenv
import os

plt.style.use('seaborn-v0_8-whitegrid')

bm_wide = pd.read_csv('bm_wide.csv', index_col='date', parse_dates=True)
ep_wide = pd.read_csv('ep_wide.csv', index_col='date', parse_dates=True)

bm_wide.index = bm_wide.index + pd.offsets.MonthEnd(0)
ep_wide.index = ep_wide.index + pd.offsets.MonthEnd(0)


### Download Macro Data from FRED

We download two variables:
- 10-year Treasury yield (GS10): when this falls, growth stocks tend to win
- VIX (VIXCLS): measures market fear — high VIX periods often hurt value stocks

In [3]:
load_dotenv()

fred = Fred(api_key=os.getenv('FRED_API_KEY'))

gs10 = fred.get_series('GS10', start='1963-01-01')
vix  = fred.get_series('VIXCLS', start='1963-01-01')

gs10_monthly = gs10.resample('ME').last()
vix_monthly  = vix.resample('ME').last()

macro = pd.DataFrame({
    'gs10': gs10_monthly,
    'vix':  vix_monthly
})

print("Date range:", macro.index.min(), "to", macro.index.max())
print(macro.tail())

Date range: 1953-04-30 00:00:00 to 2026-05-31 00:00:00
            gs10    vix
2026-01-31  4.21  17.44
2026-02-28  4.13  19.86
2026-03-31  4.25  25.25
2026-04-30  4.32  16.89
2026-05-31   NaN  18.29


### Merge Macro Data with Value Spreads

In [5]:
bm_spread = bm_wide[['spread']].copy() / 100
ep_spread = ep_wide[['spread']].copy() / 100

macro['gs10_change'] = macro['gs10'].diff()

macro = macro.dropna()

bm_macro = bm_spread.join(macro, how='inner')
ep_macro = ep_spread.join(macro, how='inner')

print("BM macro merged shape:", bm_macro.shape)
print("EP macro merged shape:", ep_macro.shape)
print(bm_macro.tail())

BM macro merged shape: (420, 4)
EP macro merged shape: (420, 4)
              spread  gs10    vix  gs10_change
2024-08-31 -0.007544  3.87  15.00        -0.38
2024-09-30 -0.003962  3.72  16.73        -0.15
2024-10-31  0.018575  4.10  23.16         0.38
2024-11-30 -0.047372  4.36  13.51         0.26
2024-12-31  0.033553  4.39  17.35         0.03


### Macro Regression: Does the Value Spread Correlate with Interest Rates and VIX?

In [6]:
X_bm = sm.add_constant(bm_macro[['gs10_change', 'vix']])
model_bm_macro = sm.OLS(bm_macro['spread'], X_bm).fit()

X_ep = sm.add_constant(ep_macro[['gs10_change', 'vix']])
model_ep_macro = sm.OLS(ep_macro['spread'], X_ep).fit()

print("=== MACRO REGRESSION RESULTS ===\n")

for name, model in [('BM', model_bm_macro), ('EP', model_ep_macro)]:
    print(f"--- {name} Spread ---")
    print(f"  Alpha (const):  {model.params['const']:.4f}  "
          f"(t = {model.tvalues['const']:.3f})")
    print(f"  GS10 Change:    {model.params['gs10_change']:.4f}  "
          f"(t = {model.tvalues['gs10_change']:.3f})")
    print(f"  VIX:            {model.params['vix']:.4f}  "
          f"(t = {model.tvalues['vix']:.3f})")
    print(f"  R-squared:      {model.rsquared:.4f}")
    print()

=== MACRO REGRESSION RESULTS ===

--- BM Spread ---
  Alpha (const):  0.0037  (t = 0.790)
  GS10 Change:    0.0200  (t = 2.640)
  VIX:            0.0000  (t = 0.033)
  R-squared:      0.0166

--- EP Spread ---
  Alpha (const):  0.0013  (t = 0.335)
  GS10 Change:    0.0029  (t = 0.456)
  VIX:            0.0000  (t = 0.123)
  R-squared:      0.0005



### Interpretation: Macro Regression Results

For BM, changes in the 10-year Treasury yield are statistically significant 
(t = 2.640). When rates rise by 1%, the BM value spread increases by 0.02 
per month. This supports the theory that falling interest rates after 2008 
contributed to value underperformance — growth stocks benefit more from low 
rates because their value comes from distant future cash flows.

VIX is not significant for either measure, suggesting market fear alone does 
not drive the value premium.

EP shows no significant relationship with either macro variable (R-squared = 0.0005),
further confirming that BM and EP behave differently.

Note: The low R-squared values (1.6% for BM, 0.05% for EP) indicate that macro 
variables explain only a small portion of the value spread — the value premium 
is driven by many factors beyond just interest rates and volatility.